# Ropedia Academy — A9 · Paper deep-dive & reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A9.ipynb)

> **A real SMPLify optimization loop plus the key ablation — turning off the pose prior fits the 2D but yields a less plausible pose (the depth-flip failure mode).**
>
> 一个真实的 SMPLify 优化循环加上关键消融——关掉姿态先验虽能拟合 2D，却得到更不合理的姿态（深度翻转失败模式）。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A9

In [ ]:
import torch

# SMPLify: FIT pose to one clip by optimization. Toy joints = base + theta (runs here).
base = torch.randn(15, 3)
joints  = lambda th: base + th
project = lambda J: J[:, :2]                      # orthographic, for brevity
kp2d = project(joints(torch.randn(15, 3))) + 0.01 * torch.randn(15, 2)   # observed 2D

def fit(prior_w):                                # minimize reprojection + pose prior
    th = torch.zeros(15, 3, requires_grad=True)
    opt = torch.optim.Adam([th], lr=0.1)
    for _ in range(300):
        opt.zero_grad()
        reproj = ((project(joints(th)) - kp2d) ** 2).mean()
        prior  = (th ** 2).mean()                 # keeps the pose plausible
        (reproj + prior_w * prior).backward(); opt.step()
    return reproj.item(), prior.item()

r1, p1 = fit(prior_w=1e-2)
r2, p2 = fit(prior_w=0.0)                          # ABLATION: remove the prior
print(f"with prior : reproj={r1:.4f}  prior_cost={p1:.3f}")
print(f"no prior   : reproj={r2:.4f}  prior_cost={p2:.3f}  <- fits 2D, less plausible pose")
print("lesson: reprojection alone is depth-ambiguous; the prior breaks the tie")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks